In [1]:
# Daca ai deja pachetele, sari peste.
# IMPORTANT: ghilimelele sunt utile ca sa nu interpreteze shell-ul parantezele patrate.
!pip install -q "emotiefflib[torch]" facenet-pytorch opencv-python pandas numpy tqdm pillow matplotlib

In [1]:
import os
import math
import json
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict

import numpy as np
import pandas as pd
from tqdm import tqdm

import cv2
from PIL import Image

import torch
from facenet_pytorch import MTCNN

import matplotlib.pyplot as plt

from emotiefflib.facial_analysis import EmotiEffLibRecognizer, get_model_list

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# === INPUT PATHS (tu ai zis ca notebook-urile sunt in /code, deci astea sunt ok) ===
SPLIT = "train"
CSV_PATH = "../Data/MELD.Raw/train/train_sent_emo.csv"
VIDEO_FOLDER = "../Data/MELD.Raw/train/train_splits"

# === OUTPUT (separat complet de ce-ai facut pana acum) ===
OUTPUT_ROOT = "../Data/processed_meld_emotieff"
RUN_NAME = "v1_enet_b2_7_gtprob_gap0p3s"   # schimba numele ca sa version-ezi usor

OUT_DIR = os.path.join(OUTPUT_ROOT, RUN_NAME)
IMAGE_OUTPUT = os.path.join(OUT_DIR, "images", SPLIT)
ANALYSIS_OUTPUT = os.path.join(OUT_DIR, "analysis", SPLIT)   # .npz per utterance cu scoruri per frame
METADATA_OUTPUT = os.path.join(OUT_DIR, f"metadata_{SPLIT}.csv")

os.makedirs(IMAGE_OUTPUT, exist_ok=True)
os.makedirs(ANALYSIS_OUTPUT, exist_ok=True)

# === selectie ===
K_FRAMES = 5                      # cate frame-uri salvezi per utterance
MIN_GAP_SECONDS = 0.3             # "nu consecutive" -> minim 0.3s intre frame-urile selectate
FRAME_STRIDE = 3                  # 1 = toate frame-urile; 2 = din 2 in 2 (daca vrei speed)

# debugging / iterare rapida
MAX_UTTERANCES = None             # ex: 50 ca test; None = ruleaza tot
SAVE_EVERY = 100                  # flush periodic la CSV (ca sa poti opri si relua)

print("Available EmotiEffLib models:", get_model_list())

Available EmotiEffLib models: ['enet_b0_8_best_vgaf', 'enet_b0_8_best_afew', 'enet_b2_8', 'enet_b0_8_va_mtl', 'enet_b2_7', 'mbf_va_mtl', 'mobilevit_va_mtl']


In [3]:
df = pd.read_csv(CSV_PATH)
print("CSV columns:", list(df.columns))
print(df.head(2))

# MELD -> normalizare label (acopera si cazul in care MELD are "joy" in loc de "happiness")
LABEL_MAP = {
    "anger": "Anger",
    "disgust": "Disgust",
    "fear": "Fear",
    "happiness": "Happiness",
    "joy": "Happiness",
    "neutral": "Neutral",
    "sadness": "Sadness",
    "surprise": "Surprise",
}

def normalize_emotion(x: str) -> str:
    x = str(x).strip().lower()
    if x not in LABEL_MAP:
        raise ValueError(f"Unknown emotion label in CSV: {x}")
    return LABEL_MAP[x]

# Coloane MELD tipice: Dialogue_ID, Utterance_ID, Emotion (uneori 'Emotion' / 'emotion')
# Le facem robust:
def get_col(name_candidates: List[str]) -> str:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in name_candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    raise KeyError(f"Missing expected column. Tried: {name_candidates}. Have: {list(df.columns)}")

COL_D = get_col(["Dialogue_ID", "dialogue_id"])
COL_U = get_col(["Utterance_ID", "utterance_id"])
COL_E = get_col(["Emotion", "emotion"])

# optional
COL_SPEAKER = None
try:
    COL_SPEAKER = get_col(["Speaker", "speaker"])
except KeyError:
    pass

df["emotion_norm"] = df[COL_E].apply(normalize_emotion)
print(df[[COL_D, COL_U, COL_E, "emotion_norm"]].head(5))

CSV columns: ['Sr No.', 'Utterance', 'Speaker', 'Emotion', 'Sentiment', 'Dialogue_ID', 'Utterance_ID', 'Season', 'Episode', 'StartTime', 'EndTime']
   Sr No.                                          Utterance          Speaker  \
0       1  also I was the point person on my companys tr...         Chandler   
1       2                   You mustve had your hands full.  The Interviewer   

   Emotion Sentiment  Dialogue_ID  Utterance_ID  Season  Episode  \
0  neutral   neutral            0             0       8       21   
1  neutral   neutral            0             1       8       21   

      StartTime       EndTime  
0  00:16:16,059  00:16:21,731  
1  00:16:21,940  00:16:23,442  
   Dialogue_ID  Utterance_ID   Emotion emotion_norm
0            0             0   neutral      Neutral
1            0             1   neutral      Neutral
2            0             2   neutral      Neutral
3            0             3   neutral      Neutral
4            0             4  surprise     Surp

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", DEVICE)

# Face detector (tu ai deja ceva similar in pipeline)
mtcnn = MTCNN(keep_all=True, device=DEVICE)

# EmotiEffLib FER model (7-class)
# Nota: enet_b2_7 este in lista lor de modele suportate :contentReference[oaicite:4]{index=4}
FER_MODEL_NAME = "enet_b2_7"
fer = EmotiEffLibRecognizer(engine="torch", model_name=FER_MODEL_NAME, device=DEVICE)

# Ordinea claselor (pt _7): 0 Anger, 1 Disgust, 2 Fear, 3 Happiness, 4 Neutral, 5 Sadness, 6 Surprise :contentReference[oaicite:5]{index=5}
IDX_TO_CLASS_7 = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
CLASS_TO_IDX_7 = {c: i for i, c in enumerate(IDX_TO_CLASS_7)}
print("Class order:", IDX_TO_CLASS_7)

Torch device: cuda
Class order: ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']


In [5]:
def resolve_video_path(dialogue_id: int, utterance_id: int, video_folder: str) -> Optional[str]:
    # Train/dev: dia{d}_utt{u}.mp4 (confirmat in tree-ul tau)
    candidates = [
        f"dia{dialogue_id}_utt{utterance_id}.mp4",
        f"final_videos_testdia{dialogue_id}_utt{utterance_id}.mp4",  # pt test (MELD are naming ciudat uneori)
    ]
    for name in candidates:
        p = os.path.join(video_folder, name)
        if os.path.exists(p):
            return p
    return None

def pick_central_box(boxes_xyxy: np.ndarray, img_w: int, img_h: int) -> Optional[np.ndarray]:
    """
    boxes_xyxy: (N,4) float, [x1,y1,x2,y2]
    Return: (4,) selected box
    """
    if boxes_xyxy is None or len(boxes_xyxy) == 0:
        return None
    cx, cy = img_w / 2.0, img_h / 2.0
    centers = np.stack([(boxes_xyxy[:,0] + boxes_xyxy[:,2]) / 2.0,
                        (boxes_xyxy[:,1] + boxes_xyxy[:,3]) / 2.0], axis=1)
    d2 = (centers[:,0] - cx)**2 + (centers[:,1] - cy)**2
    return boxes_xyxy[int(np.argmin(d2))]

def expand_and_clip_box(box: np.ndarray, img_w: int, img_h: int, scale: float = 1.2) -> Tuple[int,int,int,int]:
    x1, y1, x2, y2 = box.astype(float)
    bw, bh = x2 - x1, y2 - y1
    cx, cy = x1 + bw/2.0, y1 + bh/2.0
    bw2, bh2 = bw * scale, bh * scale
    nx1 = int(max(0, math.floor(cx - bw2/2.0)))
    ny1 = int(max(0, math.floor(cy - bh2/2.0)))
    nx2 = int(min(img_w, math.ceil(cx + bw2/2.0)))
    ny2 = int(min(img_h, math.ceil(cy + bh2/2.0)))
    if nx2 <= nx1 or ny2 <= ny1:
        return (0,0,0,0)
    return nx1, ny1, nx2, ny2

def greedy_topk_with_gap(frame_idxs: List[int], scores: List[float], k: int, min_gap_frames: int) -> List[int]:
    """
    Return indices into (frame_idxs/scores) list.
    Greedy: sort by score desc, keep if far enough from already picked.
    """
    order = np.argsort(-np.asarray(scores))
    picked = []
    for idx in order:
        fi = frame_idxs[idx]
        if all(abs(fi - frame_idxs[p]) >= min_gap_frames for p in picked):
            picked.append(int(idx))
            if len(picked) == k:
                break
    # If not enough, relax gap and just fill by score
    if len(picked) < k:
        for idx in order:
            if int(idx) not in picked:
                picked.append(int(idx))
                if len(picked) == k:
                    break
    return picked

In [6]:
# Resume support: daca ai rulat partial, sari peste utterance-urile deja procesate
processed_keys = set()
if os.path.exists(METADATA_OUTPUT):
    old = pd.read_csv(METADATA_OUTPUT)
    processed_keys = set(zip(old["dialogue_id"].astype(int), old["utterance_id"].astype(int)))
    print(f"[resume] Found {len(processed_keys)} utterances already in metadata CSV. Will skip them.")

rows_out = []  # buffer pt scriere batch in CSV

def flush_rows(rows: List[dict], out_csv: str):
    if not rows:
        return
    out_df = pd.DataFrame(rows)
    header = not os.path.exists(out_csv)
    out_df.to_csv(out_csv, mode="a", header=header, index=False)
    rows.clear()

n_done = 0
iterator = df.itertuples(index=False)
if MAX_UTTERANCES is not None:
    iterator = list(iterator)[:MAX_UTTERANCES]

for r in tqdm(iterator, total=(len(df) if MAX_UTTERANCES is None else MAX_UTTERANCES)):
    d_id = int(getattr(r, COL_D))
    u_id = int(getattr(r, COL_U))
    emo_norm = getattr(r, "emotion_norm") if hasattr(r, "emotion_norm") else normalize_emotion(getattr(r, COL_E))
    emo_idx = CLASS_TO_IDX_7[emo_norm]

    if (d_id, u_id) in processed_keys:
        continue

    video_path = resolve_video_path(d_id, u_id, VIDEO_FOLDER)
    if video_path is None:
        # video lipsa
        rows_out.append({
            "split": SPLIT, "dialogue_id": d_id, "utterance_id": u_id,
            "emotion": getattr(r, COL_E), "emotion_norm": emo_norm,
            "video_path": None, "status": "missing_video"
        })
        continue

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    min_gap_frames = max(1, int(round(fps * MIN_GAP_SECONDS)))

    frame_idxs, times_s, gt_probs = [], [], []
    pred_labels, pred_probs = [], []
    bboxes = []  # x1,y1,x2,y2

    frame_i = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        if FRAME_STRIDE > 1 and (frame_i % FRAME_STRIDE != 0):
            frame_i += 1
            continue

        # BGR -> RGB
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img_h, img_w = frame_rgb.shape[:2]

        # Face detect
        pil_img = Image.fromarray(frame_rgb)
        boxes, _ = mtcnn.detect(pil_img)  # boxes: (N,4) xyxy
        box = pick_central_box(boxes, img_w, img_h)
        if box is None:
            frame_i += 1
            continue

        x1, y1, x2, y2 = expand_and_clip_box(box, img_w, img_h, scale=1.2)
        if (x2 - x1) < 10 or (y2 - y1) < 10:
            frame_i += 1
            continue

        face_rgb = frame_rgb[y1:y2, x1:x2]  # numpy RGB

        # FER inference (probabilitati cand logits=False) :contentReference[oaicite:6]{index=6}
        # returns: (list_of_labels, scores ndarray shape (N,7))
        lbls, probs = fer.predict_emotions(face_rgb, logits=False)
        probs = probs[0]  # (7,)

        gt_p = float(probs[emo_idx])
        pred_i = int(np.argmax(probs))
        pred_lbl = IDX_TO_CLASS_7[pred_i]
        pred_p = float(probs[pred_i])

        frame_idxs.append(frame_i)
        times_s.append(frame_i / fps)
        gt_probs.append(gt_p)
        pred_labels.append(pred_lbl)
        pred_probs.append(pred_p)
        bboxes.append((x1, y1, x2, y2))

        frame_i += 1

    cap.release()

    # Daca nu ai niciun frame cu fata detectata, marchezi si sari (sau implementezi fallback)
    if len(frame_idxs) == 0:
        rows_out.append({
            "split": SPLIT, "dialogue_id": d_id, "utterance_id": u_id,
            "emotion": getattr(r, COL_E), "emotion_norm": emo_norm,
            "video_path": video_path, "fps": fps, "total_frames": total_frames,
            "status": "no_faces_detected"
        })
        processed_keys.add((d_id, u_id))
        continue

    # Salvam scorurile per frame (pentru plot ulterior)
    npz_path = os.path.join(ANALYSIS_OUTPUT, f"dia{d_id}_utt{u_id}.npz")
    np.savez_compressed(
        npz_path,
        frame_idxs=np.array(frame_idxs, dtype=np.int32),
        times_s=np.array(times_s, dtype=np.float32),
        gt_probs=np.array(gt_probs, dtype=np.float32),
        pred_probs=np.array(pred_probs, dtype=np.float32),
        # stocam pred_labels ca bytes (npz nu iubeste string-uri python direct)
        pred_labels=np.array(pred_labels, dtype="S"),
        bboxes=np.array(bboxes, dtype=np.int32),
        emotion_norm=np.array([emo_norm.encode("utf-8")]),
        video_path=np.array([video_path.encode("utf-8")]),
        fps=np.array([fps], dtype=np.float32),
    )

    # Selectie top-K cu gap
    picked_indices = greedy_topk_with_gap(frame_idxs, gt_probs, K_FRAMES, min_gap_frames)

    # Re-open video ca sa extragem exact frame-urile selectate si sa salvam crop-urile (fara re-detect)
    cap2 = cv2.VideoCapture(video_path)

    for rank, idx_in_list in enumerate(picked_indices):
        fi = frame_idxs[idx_in_list]
        t = times_s[idx_in_list]
        gt_p = gt_probs[idx_in_list]
        pr_lbl = pred_labels[idx_in_list]
        pr_p = pred_probs[idx_in_list]
        x1, y1, x2, y2 = bboxes[idx_in_list]

        cap2.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, frame_bgr = cap2.read()
        if not ok:
            continue
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        face_rgb = frame_rgb[y1:y2, x1:x2]

        # filename: include rank ca sa poti ignora 2 si folosi doar 3 dupa (ex rank 0..2)
        score_milli = int(round(gt_p * 1000))
        img_name = f"dia{d_id}_utt{u_id}_r{rank}_f{fi:05d}_gtp{score_milli:04d}.jpg"
        img_path = os.path.join(IMAGE_OUTPUT, img_name)

        # Save as JPG (RGB -> BGR for cv2.imwrite)
        cv2.imwrite(img_path, cv2.cvtColor(face_rgb, cv2.COLOR_RGB2BGR))

        rows_out.append({
            "split": SPLIT,
            "dialogue_id": d_id,
            "utterance_id": u_id,
            "speaker": (getattr(r, COL_SPEAKER) if COL_SPEAKER is not None else None),
            "emotion": getattr(r, COL_E),
            "emotion_norm": emo_norm,
            "video_path": video_path,
            "fps": fps,
            "total_frames": total_frames,
            "frame_idx": fi,
            "time_s": t,
            "selection_rank": rank,
            "gt_prob": gt_p,
            "pred_label": pr_lbl,
            "pred_prob": pr_p,
            "bbox_x1": x1, "bbox_y1": y1, "bbox_x2": x2, "bbox_y2": y2,
            "image_path": img_path,
            "analysis_npz": npz_path,
            "status": "ok"
        })

    cap2.release()

    processed_keys.add((d_id, u_id))
    n_done += 1

    if n_done % SAVE_EVERY == 0:
        flush_rows(rows_out, METADATA_OUTPUT)

# flush final
flush_rows(rows_out, METADATA_OUTPUT)
print("DONE. Metadata:", METADATA_OUTPUT)
print("Images:", IMAGE_OUTPUT)
print("Analysis npz:", ANALYSIS_OUTPUT)

[resume] Found 9522 utterances already in metadata CSV. Will skip them.


100%|██████████| 9989/9989 [29:12<00:00,  5.70it/s] 

DONE. Metadata: ../Data/processed_meld_emotieff\v1_enet_b2_7_gtprob_gap0p3s\metadata_train.csv
Images: ../Data/processed_meld_emotieff\v1_enet_b2_7_gtprob_gap0p3s\images\train
Analysis npz: ../Data/processed_meld_emotieff\v1_enet_b2_7_gtprob_gap0p3s\analysis\train
